# mlplatform - Local Workflow

This notebook walks through the full local workflow:
1. **Setup** - Configure paths and PYTHONPATH
2. **Training** - Run training with evaluation and artifact persistence
3. **Artifact Inspection** - View saved artifacts via `ArtifactRegistry`
4. **In-process Prediction** - Framework-managed data I/O + `predict_chunk`
5. **SQL Templates** - Parameterised BigQuery queries for data loading
6. **PySpark Batch Prediction** - Distributed prediction via `mapInPandas`
7. **Cleanup**

**Prerequisites:** `pyyaml`, `scikit-learn`, `pandas`, `joblib`, `pyspark`, `pyarrow` must be installed.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("REPO_ROOT", Path("..").resolve())).resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "mlplatform") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "mlplatform"))

VERSION = "notebook_v1"
BASE_PATH = str(REPO_ROOT / "artifacts_notebook")
TRAIN_DAG = str(REPO_ROOT / "example_model" / "pipeline" / "dags" / "train.yaml")
PREDICT_DAG = str(REPO_ROOT / "example_model" / "pipeline" / "dags" / "predict.yaml")

print(f"REPO_ROOT:    {REPO_ROOT}")
print(f"VERSION:      {VERSION}")
print(f"BASE_PATH:    {BASE_PATH}")
print(f"TRAIN_DAG:    {TRAIN_DAG}")
print(f"PREDICT_DAG:  {PREDICT_DAG}")

## 2. Training

Load the training DAG config, build an `ExecutionContext` (with `ArtifactRegistry`), and run the trainer.

The trainer (`example_model.train.MyTrainer`) will:
- Load data from CSV (or BigQuery when `bq_params` is set in YAML)
- Split into train/validation, fit StandardScaler + LogisticRegression
- Evaluate with `example_model.evaluate.evaluate()` (accuracy, precision, recall, F1)
- Save artifacts via `ctx.save_artifact()` (delegates to `ArtifactRegistry`)
- Log metrics via the experiment tracker

In [ ]:
import pandas as pd
from mlplatform.config.loader import load_workflow_config
from mlplatform.runner import _build_context, _run_training, _log_framework_params

workflow = load_workflow_config(TRAIN_DAG)
model_cfg = workflow.models[0]

print(f"Workflow:      {workflow.workflow_name}")
print(f"Pipeline type: {workflow.pipeline_type}")
print(f"Feature name:  {workflow.feature_name}")
print(f"Model name:    {model_cfg.model_name}")
print(f"Module:        {model_cfg.module}")

In [ ]:
PROFILE = "local"
COMMIT_HASH = "abc1234"  # pass real hash via git rev-parse HEAD

ctx = _build_context(workflow, model_cfg, PROFILE, VERSION, BASE_PATH, commit_hash=COMMIT_HASH)
_log_framework_params(ctx, PROFILE)

_run_training(model_cfg, ctx)
print("Training complete.")

## 3. Artifact Inspection

The `ArtifactRegistry` stores artifacts under `{feature}/{model}/{version}/`.

In [ ]:
import json

base = Path(BASE_PATH)
print("Saved artifacts:")
for p in sorted(base.rglob("*.pkl")):
    print(f"  {p.relative_to(base)}")

print(f"\nArtifactRegistry base_path: {ctx.artifacts.base_path}")
print(f"Storage backend:           {type(ctx.storage).__name__}")

tracker_path = base / "tracking"
if tracker_path.exists():
    for p in sorted(tracker_path.rglob("*.json")):
        print(f"\nTracking log: {p.relative_to(base)}")
        with open(p) as f:
            print(json.dumps(json.load(f), indent=2))

## 4. In-process Prediction

The predictor focuses on model logic only. Data I/O is managed by the
`InvocationStrategy`. Here we manually load the predictor and call
`predict_chunk` to show the low-level API, then demonstrate the full
`InProcessInvocation` flow.

In [ ]:
pred_workflow = load_workflow_config(PREDICT_DAG)
pred_model_cfg = pred_workflow.models[0]
pred_ctx = _build_context(pred_workflow, pred_model_cfg, PROFILE, VERSION, BASE_PATH)

from example_model.predict import MyPredictor

predictor = MyPredictor()
predictor.context = pred_ctx
predictor.load_model()
print("Model loaded via ArtifactRegistry.")

In [ ]:
from sklearn.datasets import make_classification

X_test, _ = make_classification(n_samples=10, n_features=5, random_state=99)
test_df = pd.DataFrame(X_test, columns=["f0", "f1", "f2", "f3", "f4"])

result = predictor.predict_chunk(test_df)
result

### Prediction with the sample CSV from example_model/data/

In [ ]:
inference_csv = REPO_ROOT / "example_model" / "data" / "sample_inference.csv"
inference_df = pd.read_csv(inference_csv)
print(f"Input ({len(inference_df)} rows):")
display(inference_df)

result_csv = predictor.predict_chunk(inference_df)
print(f"\nPredictions:")
display(result_csv)

## 5. SQL Templates

Data scientists define parameterised SQL in `example_model/sql/`. Parameters
from `optional_configs.bq_params` are injected at runtime.  The rendered SQL
is passed to BigQuery (in production) or previewed here.

In [ ]:
from example_model.utils import load_sql_template, render_sql

train_sql_raw = load_sql_template("load_training_data.sql")
print("=== Raw template ===")
print(train_sql_raw)

bq_params = {
    "project": "my-gcp-project",
    "dataset": "my_dataset",
    "table": "training_features",
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "sample_fraction": 1.0,
    "target_column": "target",
}
rendered = render_sql(train_sql_raw, bq_params)
print("=== Rendered SQL ===")
print(rendered)

In [ ]:
pred_sql_raw = load_sql_template("load_prediction_data.sql")
pred_bq_params = {
    "project": "my-gcp-project",
    "dataset": "my_dataset",
    "table": "scoring_features",
    "scoring_date": "2025-06-01",
    "unique_id_column": "vin",
}
print("=== Rendered prediction SQL ===")
print(render_sql(pred_sql_raw, pred_bq_params))

In [ ]:
## 6. PySpark Batch Prediction

This demonstrates the Spark `mapInPandas` flow used for distributed prediction.
The same code runs locally (here) and on cloud (Dataproc/VertexAI).

Steps:
1. Serialize the prediction config to JSON (as `spark/main.py` expects)
2. Create an input CSV for Spark to read
3. `_run_spark_inference()` starts a local Spark session and delegates to `SparkBatchInvocation`

In [ ]:
from mlplatform.spark.config_serializer import workflow_config_to_dict

dist_dir = REPO_ROOT / "dist_notebook"
dist_dir.mkdir(exist_ok=True)

spark_csv_path = str(dist_dir / "spark_input.csv")
spark_output_path = str(dist_dir / "predictions.parquet")

X_spark, _ = make_classification(n_samples=20, n_features=5, random_state=77)
spark_input = pd.DataFrame(X_spark, columns=["f0", "f1", "f2", "f3", "f4"])
spark_input.to_csv(spark_csv_path, index=False)
print(f"Input CSV: {spark_csv_path} ({len(spark_input)} rows)")

spark_config = workflow_config_to_dict(
    pred_workflow, pred_model_cfg,
    base_path=BASE_PATH, version=VERSION, commit_hash=COMMIT_HASH,
)
spark_config["runtime_config"]["input_path"] = spark_csv_path
spark_config["runtime_config"]["output_path"] = spark_output_path

print("\nSpark config:")
print(json.dumps(spark_config, indent=2))

### Run Spark inference

In [ ]:
from mlplatform.spark.main import _run_spark_inference

_run_spark_inference(spark_config)
print("Spark inference complete.")

In [ ]:
spark_result = pd.read_parquet(spark_output_path)
print(f"Output: {len(spark_result)} rows, columns: {list(spark_result.columns)}")
spark_result.head(10)

## 7. Cleanup

Remove the artifacts and dist directories created during this notebook run.

In [ ]:
import shutil

for d in [Path(BASE_PATH), dist_dir]:
    if d.exists():
        shutil.rmtree(d)
        print(f"Removed {d}")

print("Cleanup complete.")